In [ ]:
CODE UTAMA

In [ ]:
# =========================================================
# PIANO HAND GESTURE RECOGNITION WITH LSTM
# =========================================================
# FITUR:
# ✅ MediaPipe Hands
# ✅ Feature Engineering (sudut + jarak + landmark)
# ✅ Sliding Window Sequence
# ✅ LSTM
# ✅ K-Fold Cross Validation
# ✅ Hyperparameter Tuning (Keras Tuner)
# ✅ Fine Tuning
# ✅ Save Best Model
# ✅ Confusion Matrix
# ✅ Classification Report (4 decimal)
# =========================================================

# =========================================================
# 1. IMPORT LIBRARY
# =========================================================
import os
import gc
import cv2
import keras_tuner as kt
import numpy as np
import mediapipe as mp
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.model_selection import KFold
from sklearn.metrics import (
    confusion_matrix,
    classification_report
)

from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

from tensorflow.keras.utils import to_categorical

# =========================================================
# 2. OUTPUT DIRECTORY
# =========================================================
OUTPUT_DIR = "OUTPUT_PIANO_LSTM"

CM_DIR = os.path.join(OUTPUT_DIR, "confusion_matrix")
REPORT_DIR = os.path.join(OUTPUT_DIR, "classification_report")
MODEL_DIR = os.path.join(OUTPUT_DIR, "best_model")
HP_DIR = os.path.join(OUTPUT_DIR, "best_hyperparameter")

os.makedirs(CM_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(HP_DIR, exist_ok=True)

# =========================================================
# 3. MEDIAPIPE SETUP
# =========================================================
mp_hands = mp.solutions.hands

# =========================================================
# 4. PARAMETER
# =========================================================
DATASET_PATH = "Dataset"

SEQUENCE_LENGTH = 30
STRIDE = 5

EPOCHS = 50
BATCH_SIZE = 8

# =========================================================
# LABEL
# =========================================================
label_map = {
    "good": 0,
    "needs_improvement": 1,
    "poor": 2
}

class_names = list(label_map.keys())

N_CLASSES = len(label_map)

# =========================================================
# 5. DEFINISI SENDI
# =========================================================
JOINTS = [
    [2, 1, 3],
    [3, 2, 4],

    [6, 5, 7],
    [7, 6, 8],

    [10, 9, 11],
    [11, 10, 12],

    [14, 13, 15],
    [15, 14, 16],

    [18, 17, 19],
    [19, 18, 20]
]

# =========================================================
# 6. HITUNG SUDUT
# =========================================================
def calculate_angle(a, b, c):

    a = np.array(a)
    b = np.array(b)
    c = np.array(c)

    ba = a - b
    bc = c - b

    cosine = np.dot(ba, bc) / (
        np.linalg.norm(ba) *
        np.linalg.norm(bc) + 1e-6
    )

    cosine = np.clip(cosine, -1.0, 1.0)

    angle = np.degrees(np.arccos(cosine))

    return angle

# =========================================================
# 7. EXTRACT FEATURE PER FRAME
# =========================================================
def extract_frame_features(hand_landmarks):

    features = []

    coords = []

    # =====================================================
    # RAW LANDMARK
    # =====================================================
    for lm in hand_landmarks.landmark:

        coords.append([lm.x, lm.y, lm.z])

        features.extend([lm.x, lm.y, lm.z])

    coords = np.array(coords)

    # =====================================================
    # ANGLE FEATURE
    # =====================================================
    for joint in JOINTS:

        p1 = coords[joint[0]]
        p2 = coords[joint[1]]
        p3 = coords[joint[2]]

        angle = calculate_angle(p1, p2, p3)

        features.append(angle)

    # =====================================================
    # DISTANCE FEATURE
    # =====================================================
    fingertips = [4, 8, 12, 16, 20]

    for i in range(len(fingertips)):
        for j in range(i + 1, len(fingertips)):

            p1 = coords[fingertips[i]]
            p2 = coords[fingertips[j]]

            dist = np.linalg.norm(p1 - p2)

            features.append(dist)

    return features

# =========================================================
# 8. EXTRACT VIDEO
# =========================================================
def extract_video_sequences(video_path):

    cap = cv2.VideoCapture(video_path)

    all_frames = []

    with mp_hands.Hands(
        static_image_mode=False,
        max_num_hands=1,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    ) as hands:

        while cap.isOpened():

            ret, frame = cap.read()

            if not ret:
                break

            image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            results = hands.process(image)

            if results.multi_hand_landmarks:

                hand_landmarks = results.multi_hand_landmarks[0]

                features = extract_frame_features(hand_landmarks)

            else:

                features = [0] * 83

            all_frames.append(features)

    cap.release()

    # =====================================================
    # SLIDING WINDOW
    # =====================================================
    sequences = []

    for i in range(
        0,
        len(all_frames) - SEQUENCE_LENGTH,
        STRIDE
    ):

        seq = all_frames[i:i + SEQUENCE_LENGTH]

        sequences.append(seq)

    return np.array(sequences)

# =========================================================
# 9. LOAD DATASET
# =========================================================
X = []
y = []

print("=" * 60)
print("LOADING DATASET...")
print("=" * 60)

for label in label_map:

    folder = os.path.join(DATASET_PATH, label)

    if not os.path.exists(folder):
        continue

    for file in os.listdir(folder):

        if file.lower().endswith((".mp4", ".mov")):

            path = os.path.join(folder, file)

            print("Processing:", path)

            sequences = extract_video_sequences(path)

            for seq in sequences:

                X.append(seq)
                y.append(label_map[label])

# =========================================================
# CONVERT
# =========================================================
X = np.array(X)

y = np.array(y)

y = to_categorical(y, num_classes=N_CLASSES)

# =========================================================
# INFO
# =========================================================
print("\n")
print("=" * 60)
print("DATASET INFO")
print("=" * 60)

print("X shape:", X.shape)
print("y shape:", y.shape)

FEATURES = X.shape[2]

print("Features per frame:", FEATURES)

# =========================================================
# 10. MODEL BUILDER (KERAS TUNER)
# =========================================================
def model_builder(hp):

    model = models.Sequential()

    # =====================================================
    # HYPERPARAMETER
    # =====================================================
    lstm_1 = hp.Int(
        "lstm_1",
        min_value=64,
        max_value=256,
        step=64
    )

    lstm_2 = hp.Int(
        "lstm_2",
        min_value=32,
        max_value=128,
        step=32
    )

    dense_units = hp.Int(
        "dense_units",
        min_value=32,
        max_value=256,
        step=32
    )

    dropout_rate = hp.Float(
        "dropout",
        0.2,
        0.5,
        step=0.1
    )

    learning_rate = hp.Choice(
        "learning_rate",
        [1e-3, 1e-4, 1e-5]
    )

    # =====================================================
    # LSTM 1
    # =====================================================
    model.add(
        layers.LSTM(
            lstm_1,
            return_sequences=True,
            input_shape=(SEQUENCE_LENGTH, FEATURES)
        )
    )

    model.add(layers.Dropout(dropout_rate))

    # =====================================================
    # LSTM 2
    # =====================================================
    model.add(
        layers.LSTM(lstm_2)
    )

    model.add(layers.Dropout(dropout_rate))

    # =====================================================
    # DENSE
    # =====================================================
    model.add(
        layers.Dense(
            dense_units,
            activation="relu",
            kernel_regularizer=regularizers.l2(1e-4)
        )
    )

    model.add(layers.Dropout(dropout_rate))

    # =====================================================
    # OUTPUT
    # =====================================================
    model.add(
        layers.Dense(
            N_CLASSES,
            activation="softmax"
        )
    )

    # =====================================================
    # COMPILE
    # =====================================================
    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=learning_rate
        ),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

# =========================================================
# SAVE CONFUSION MATRIX
# =========================================================
def save_confusion_matrix(cm, class_names, title, filename):

    plt.figure(figsize=(6, 5))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names
    )

    plt.title(title)

    plt.xlabel("Predicted")
    plt.ylabel("True")

    plt.tight_layout()

    plt.savefig(filename)

    plt.close()

# =========================================================
# 11. K-FOLD
# =========================================================
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

all_true = []
all_pred = []

# =========================================================
# TRAINING
# =========================================================
for fold, (train_idx, val_idx) in enumerate(kf.split(X)):

    print("\n")
    print("=" * 60)
    print(f"FOLD {fold+1}/5")
    print("=" * 60)

    X_train = X[train_idx]
    X_val   = X[val_idx]

    y_train = y[train_idx]
    y_val   = y[val_idx]

    # =====================================================
    # CLASS WEIGHT
    # =====================================================
    y_int = np.argmax(y_train, axis=1)

    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.unique(y_int),
        y=y_int
    )

    class_weights = {
        i: class_weights[i]
        for i in range(N_CLASSES)
    }

    # =====================================================
    # TUNER
    # =====================================================
    tuner = kt.RandomSearch(
        model_builder,
        objective="val_accuracy",
        max_trials=5,
        executions_per_trial=1,
        directory="Tuner_Result",
        project_name=f"Fold_{fold+1}",
        overwrite=True
    )

    tuner.search(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        class_weight=class_weights,
        callbacks=[
            EarlyStopping(
                monitor="val_loss",
                patience=5,
                restore_best_weights=True
            ),

            ReduceLROnPlateau(
                monitor="val_loss",
                factor=0.5,
                patience=2
            )
        ],
        verbose=1
    )

    # =====================================================
    # BEST HYPERPARAMETER
    # =====================================================
    best_hp = tuner.get_best_hyperparameters(1)[0]

    hp_path = os.path.join(
        HP_DIR,
        f"best_hyperparameter_fold_{fold+1}.txt"
    )

    with open(hp_path, "w") as f:

        f.write(f"=== BEST HYPERPARAMETER FOLD {fold+1} ===\n\n")

        for key in best_hp.values.keys():

            f.write(f"{key}: {best_hp.get(key)}\n")

    # =====================================================
    # BEST MODEL
    # =====================================================
    best_model = tuner.get_best_models(1)[0]

    model_path = os.path.join(
        MODEL_DIR,
        f"best_model_fold_{fold+1}.h5"
    )

    best_model.save(model_path)

    # =====================================================
    # PREDICTION
    # =====================================================
    preds = best_model.predict(X_val)

    y_pred = np.argmax(preds, axis=1)

    y_true = np.argmax(y_val, axis=1)

    # =====================================================
    # SAVE TOTAL
    # =====================================================
    all_true.extend(y_true)
    all_pred.extend(y_pred)

    # =====================================================
    # CONFUSION MATRIX
    # =====================================================
    cm = confusion_matrix(
        y_true,
        y_pred
    )

    cm_path = os.path.join(
        CM_DIR,
        f"cm_fold_{fold+1}.png"
    )

    save_confusion_matrix(
        cm,
        class_names,
        title=f"Confusion Matrix Fold {fold+1}",
        filename=cm_path
    )

    # =====================================================
    # CLASSIFICATION REPORT
    # =====================================================
    report = classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        digits=4
    )

    report_path = os.path.join(
        REPORT_DIR,
        f"classification_report_fold_{fold+1}.txt"
    )

    with open(report_path, "w") as f:

        f.write(report)

    print("\n")
    print(report)

    # =====================================================
    # CLEAR SESSION
    # =====================================================
    tf.keras.backend.clear_session()

    gc.collect()

# =========================================================
# TOTAL CONFUSION MATRIX
# =========================================================
cm_total = confusion_matrix(
    all_true,
    all_pred
)

cm_total_path = os.path.join(
    CM_DIR,
    "cm_total.png"
)

save_confusion_matrix(
    cm_total,
    class_names,
    title="Confusion Matrix Total",
    filename=cm_total_path
)

# =========================================================
# TOTAL CLASSIFICATION REPORT
# =========================================================
report_total = classification_report(
    all_true,
    all_pred,
    target_names=class_names,
    digits=4
)

total_report_path = os.path.join(
    REPORT_DIR,
    "classification_report_total.txt"
)

with open(total_report_path, "w") as f:

    f.write(
        "=== CLASSIFICATION REPORT TOTAL ===\n\n"
    )

    f.write(report_total)

print("\n")
print("=" * 60)
print("FINAL REPORT")
print("=" * 60)

print(report_total)

print("\n")
print("SEMUA PROSES SELESAI!")

Trial 3 Complete [00h 05m 15s]
val_accuracy: 0.6474359035491943

Best val_accuracy So Far: 0.6730769276618958
Total elapsed time: 00h 13m 38s

Search: Running Trial #4

Value             |Best Value So Far |Hyperparameter
128               |128               |lstm_1
96                |128               |lstm_2
256               |224               |dense_units
0.2               |0.3               |dropout
0.001             |0.001             |learning_rate

Epoch 1/50
  3/235 [..............................] - ETA: 11s - loss: 1.1421 - accuracy: 0.3750  

In [ ]:
#Mencoba mediapipe

In [5]:
# =========================
# 1. INSTALL (Colab only)
# =========================
#!pip install mediapipe opencv-python tensorflow scikit-learn numpy

# =========================
# 2. IMPORT
# =========================
import os
import cv2
import numpy as np
import mediapipe as mp

# from sklearn.model_selection import train_test_split
# from tensorflow.keras.utils import to_categorical
# from tensorflow.keras.models import Sequential
# from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional
# from tensorflow.keras.callbacks import EarlyStopping

# =========================
# 3. SETUP MEDIAPIPE
# =========================
mp_hands = mp.solutions.hands

# =========================
# 4. PARAMETER
# =========================
DATASET_PATH = "Dataset"   # ganti sesuai lokasi kamu
MAX_FRAMES = 30
FEATURES = 21 * 3 * 2  # 2 tangan

label_map = {
    "good": 0,
    "needs_improvement": 1,
    "poor": 2
}

# =========================
# 5. EXTRACT KEYPOINTS
# =========================
def extract_keypoints(video_path):
    cap = cv2.VideoCapture(video_path)

    all_frames = []

    with mp_hands.Hands(
        static_image_mode=False,
        max_num_hands=2,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    ) as hands:

        while cap.isOpened():
            ret, frame = cap.read()

            if not ret:
                break

            image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = hands.process(image)

            keypoints = []

            if results.multi_hand_landmarks:

                for hand_landmarks in results.multi_hand_landmarks:
                    for lm in hand_landmarks.landmark:
                        keypoints.extend([lm.x, lm.y, lm.z])

                # padding jika hanya 1 tangan
                if len(results.multi_hand_landmarks) == 1:
                    keypoints.extend([0] * (21 * 3))

            else:
                keypoints = [0] * FEATURES

            all_frames.append(keypoints)

    cap.release()

    # =========================
    # CREATE SEQUENCES
    # =========================
    sequences = []

    SEQ_LEN = 30
    STRIDE = 5

    for i in range(0, len(all_frames) - SEQ_LEN, STRIDE):
        sequence = all_frames[i:i + SEQ_LEN]
        sequences.append(sequence)

    return np.array(sequences)
    
# =========================
# 6. LOAD DATASET
# =========================
X = []
y = []

for label in label_map:

    folder = os.path.join(DATASET_PATH, label)

    for file in os.listdir(folder):

        if file.endswith(".mp4"):

            path = os.path.join(folder, file)

            print("Processing:", path)

            sequences = extract_keypoints(path)

            for seq in sequences:
                X.append(seq)
                y.append(label_map[label])

X = np.array(X)

print("X shape:", X.shape)
print("y shape:", len(y))



Processing: Dataset\needs_improvement\WhatsApp Video 2026-05-25 at 15.50.04.mp4
Processing: Dataset\poor\good.mp4
X shape: (72, 30, 126)
y shape: 72


In [ ]:
#Mencoba Tensorflow

In [1]:
import cv2
import mediapipe as mp
import matplotlib.pyplot as plt

VIDEO_PATH = "Dataset/poor/video1.mp4"

cap = cv2.VideoCapture(VIDEO_PATH)

ret, frame = cap.read()
cap.release()

frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

mp_hands = mp.solutions.hands
mp_draw = mp.solutions.drawing_utils

with mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=1,
    min_detection_confidence=0.5
) as hands:

    results = hands.process(frame_rgb)

    if results.multi_hand_landmarks:

        for hand_landmarks in results.multi_hand_landmarks:

            mp_draw.draw_landmarks(
                frame_rgb,
                hand_landmarks,
                mp_hands.HAND_CONNECTIONS
            )

plt.figure(figsize=(8,6))
plt.imshow(frame_rgb)
plt.axis("off")
plt.title("MediaPipe Hand Landmark Detection")
plt.savefig("gambar_landmark_tangan.png", dpi=300)
plt.show()

error: OpenCV(4.10.0) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\color.cpp:196: error: (-215:Assertion failed) !_src.empty() in function 'cv::cvtColor'
